# SSTZIP-GNN Training Notebook

Train SSTZIP-GNN model on all 4 clustering methods (Baseline, Method1, Method2, Method3)

**Works on:**
- Local machine (CPU/GPU)
- Google Colab (GPU T4/A100)

**Steps:**
1. Setup environment & check GPU
2. Load configuration
3. Import project modules
4. Run training pipeline
5. Display results

## 1. Environment Setup

In [1]:
!git checkout develop

fatal: not a git repository (or any of the parent directories): .git


In [1]:
# Check if running on Colab
import sys
import os
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
    print("[INFO] Running on Google Colab")
except ImportError:
    IN_COLAB = False
    print("[INFO] Running locally")

# Setup environment
if IN_COLAB:
    # Mount Google Drive
    drive.mount('/content/drive', force_remount=False)
    print("[OK] Google Drive mounted")
    
    # Clone repository
    print("[Clone] Cloning repository...")
    !git clone -b develop https://github.com/senkochi/taxi-demand-prediction.git /content/taxi-demand-prediction 2>/dev/null || echo "Repository already cloned"
    
    os.chdir('/content/taxi-demand-prediction')
    print(f"[OK] Working directory: {os.getcwd()}")
    
    # Link data from Google Drive
    print("\n[Link] Linking data from Google Drive...")
    drive_data = Path('/content/drive/MyDrive/data')
    local_data = Path('data')
    
    if not local_data.exists():
        try:
            os.symlink(drive_data, 'data')
            print(f"  [OK] Symlink: ./data → {drive_data}")
        except (OSError, NotImplementedError):
            import shutil
            shutil.copytree(drive_data, 'data')
            print(f"  [OK] Data copied from Google Drive")
    else:
        print(f"  [OK] ./data already exists")
else:
    # Local setup - navigate to project root
    notebook_dir = Path.cwd()
    
    # If in notebooks folder, go up to project root
    if notebook_dir.name == 'notebooks':
        project_root = notebook_dir.parent
    elif (notebook_dir / 'notebooks').exists():
        project_root = notebook_dir
    else:
        project_root = notebook_dir.parent
    
    os.chdir(project_root)
    print(f"[OK] Working directory: {os.getcwd()}")

# Verify structure
print("\n[Check] Project structure:")
for item in ['src', 'config', 'scripts', 'data']:
    exists = Path(item).exists()
    status = "✓" if exists else "✗"
    print(f"  {status} {item}/")

Mounted at /content/drive
[OK] Google Drive mounted
[Clone] Cloning repository...
[OK] Working directory: /content/taxi-demand-prediction

[Link] Linking data from Google Drive...
  [OK] Symlink: ./data → /content/drive/MyDrive/data

[Check] Project structure:
  ✓ src/
  ✓ config/
  ✓ scripts/
  ✓ data/


In [2]:
# Install dependencies on Colab
if IN_COLAB:
    print("[Install] Installing PyTorch Lightning...")
    !pip install pytorch-lightning -q
    !pip install duckdb -q
    !pip install pyyaml -q
    print("[OK] Dependencies installed")

[Install] Installing PyTorch Lightning...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 14.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 33.9 MB/s eta 0:00:00
[OK] Dependencies installed


In [3]:
# Check GPU availability
import torch

print("[System] PyTorch environment:")
print(f"  - Version: {torch.__version__}")
print(f"  - CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  - GPU: {torch.cuda.get_device_name(0)}")
    print(f"  - VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = 'cuda'
else:
    print(f"  - Using CPU")
    device = 'cpu'

[System] PyTorch environment:
  - Version: 2.11.0+cpu
  - CUDA available: False
  - Using CPU


## 2. Load Configuration

In [4]:
import yaml
from pathlib import Path
import os

# Get actual current working directory
cwd = os.getcwd()
print(f"[Debug] Current working directory: {cwd}\n")

# Try multiple possible config paths
config_paths = [
    Path('config/config.yaml'),  # Relative to cwd
    Path.cwd() / 'config' / 'config.yaml',  # Absolute from cwd
]

# If running from notebooks folder, try parent directory
if Path.cwd().name == 'notebooks' or 'notebooks' in str(Path.cwd()):
    project_root = Path.cwd().parent
    config_paths.insert(0, project_root / 'config' / 'config.yaml')
    print(f"[Debug] Detected notebooks folder, also checking: {config_paths[0]}\n")

# Find config
config_path = None
for path in config_paths:
    if path.exists():
        config_path = path
        print(f"[OK] Found config at: {path}")
        break

if not config_path:
    print(f"[ERROR] Config not found in any of these locations:")
    for path in config_paths:
        print(f"  - {path}")
    print(f"\nCurrent working directory: {cwd}")
    print(f"Looking for: config/config.yaml")
    raise FileNotFoundError(f"Config file not found")

# Load configuration
with open(config_path) as f:
    config = yaml.safe_load(f)

print("\n[Config] Loaded configuration:")
print(f"  - Architecture: {config.get('model', {}).get('architecture', 'SSTZIP-GNN')}")
print(f"  - Methods: {config.get('clustering', {}).get('methods', [])}")
print(f"  - Time buckets: {config.get('temporal_aggregation', {}).get('buckets', [])}")
print(f"  - Batch size: {config.get('model', {}).get('training', {}).get('batch_size', 64)}")
print(f"  - Epochs: {config.get('model', {}).get('training', {}).get('epochs', 100)}")
print(f"  - Learning rate: {config.get('model', {}).get('training', {}).get('learning_rate', 0.001)}")

[Debug] Current working directory: /content/taxi-demand-prediction

[OK] Found config at: config/config.yaml

[Config] Loaded configuration:
  - Architecture: SSTZIP-GNN
  - Methods: ['baseline', 'method1', 'method2', 'method3']
  - Time buckets: [15, 30, 60]
  - Batch size: 64
  - Epochs: 100
  - Learning rate: 0.001


## 3. Import Project Modules

In [5]:
import sys
from pathlib import Path

# Add project to path
sys.path.insert(0, str(Path.cwd()))

# Import project modules
try:
    from src.data.data_loader import TaxiDemandDataModule
    from src.models.sstzip_gnn import SSTZIPGNNModel
    from src.training.trainer import SSTZIPGNNLightning
    from src.evaluation.metrics import Metrics
    print("[OK] All project modules imported successfully!")
except ImportError as e:
    print(f"[ERROR] Failed to import project modules: {e}")
    print("\nAvailable modules in src/:")
    import os
    for item in os.listdir('src'):
        print(f"  - {item}")
    raise

[OK] All project modules imported successfully!


## 4. Verify Required Data Files

In [9]:
from pathlib import Path

print("[Check] Verifying required data files...\n")

# Check DuckDB features file
duckdb_path = Path('data/processed/taxi_features.duckdb')
if duckdb_path.exists():
    size_mb = duckdb_path.stat().st_size / 1e6
    print(f"  [OK] DuckDB: {duckdb_path} ({size_mb:.1f} MB)")
else:
    print(f"  [MISSING] DuckDB: {duckdb_path}")

# Check cluster assignment files
print("\n[Check] Cluster assignment files:")
cluster_files = {
    'Baseline': Path('data/models/baseline_clusters.pkl'),
    'Method1': Path('data/models/method1_clusters.pkl'),
    'Method2': Path('data/models/method2_clusters.pkl'),
    'Method3': Path('data/models/method3_clusters.pkl'),
}

missing_clusters = []
for name, path in cluster_files.items():
    if path.exists():
        print(f"  [OK] {name}: {path}")
    else:
        print(f"  [MISSING] {name}: {path}")
        missing_clusters.append(name)

if missing_clusters:
    print(f"\n[WARN] Missing cluster files for: {', '.join(missing_clusters)}")
    print("  These should be generated by Phase 3 (clustering scripts)")
else:
    print("\n[OK] All cluster files present!")

print("\n[Note] Data structure expected:")
print("  My Drive/data/")
print("    ├── processed/")
print("    │   ├── taxi_features.duckdb")
print("    │   ├── taxi_features_*.parquet")
print("    │   └── *_report.json")
print("    └── models/")
print("        └── sstzip_gnn/")
print("            ├── baseline_clusters.pkl")
print("            ├── method1_clusters.pkl")
print("            ├── method2_clusters.pkl")
print("            └── method3_clusters.pkl")

[Check] Verifying required data files...

  [OK] DuckDB: data/processed/taxi_features.duckdb (489.7 MB)

[Check] Cluster assignment files:
  [MISSING] Baseline: data/models/baseline_clusters.pkl
  [OK] Method1: data/models/method1_clusters.pkl
  [OK] Method2: data/models/method2_clusters.pkl
  [OK] Method3: data/models/method3_clusters.pkl

[WARN] Missing cluster files for: Baseline
  These should be generated by Phase 3 (clustering scripts)

[Note] Data structure expected:
  My Drive/data/
    ├── processed/
    │   ├── taxi_features.duckdb
    │   ├── taxi_features_*.parquet
    │   └── *_report.json
    └── models/
        └── sstzip_gnn/
            ├── baseline_clusters.pkl
            ├── method1_clusters.pkl
            ├── method2_clusters.pkl
            └── method3_clusters.pkl


## 5. Run Training Script

In [ ]:
# Execute the training script
print("="*80)
print("EXECUTING: scripts/04_train_model.py")
print("="*80)
print()

# Read and execute the training script
import os
script_path = os.path.join(os.getcwd(), 'scripts/04_train_model.py')

with open(script_path, encoding='utf-8') as f:
    training_script = f.read()

# Set __file__ for the script namespace
exec_globals = {'__file__': script_path, '__name__': '__main__'}
exec(training_script, exec_globals)

EXECUTING: scripts/04_train_model.py


PHASE 4: DEEP LEARNING MODELING - SSTZIP-GNN TRAINING

[Setup] Loading configuration...

[Data] Initializing data module and loading adjacency matrices...

--------------------------------------------------------------------------------
METHOD: BASELINE
--------------------------------------------------------------------------------
Setting up data module for baseline...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 3513833 feature records from 261 zones

⚠️  Skipping baseline: Clustering file not found: data/models/baseline_clusters.pkl
   Make sure clustering results exist in data/models/

--------------------------------------------------------------------------------
METHOD: METHOD1
--------------------------------------------------------------------------------
Setting up data module for method1...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 3513833 feature records from 261 zones
Adjacency matrix created: torch.Size([264, 264])
Train: 2369537 sequences
Val: 251674 sequences
Test: 868533 sequences
Creating adjacency matrix from method1 clustering...
✓ Adjacency matrix: torch.Size([264, 264])

TRAINING SSTZIP-GNN: METHOD1

[Config] Sequence length: 96
[Config] Batch size: 64
[Config] Learning rate: 0.001
[Config] Epochs: 100
[1/7] Getting data loaders for method1...
✓ DataLoaders ready
  - Train batches: 37025
  - Val batches: 3933
  - Test batches: 13571
  - Adjacency matrix: torch.Size([264, 264])

[2/7] Initializing SSTZIP-GNN model...


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 3513833 feature records from 261 zones
Adjacency matrix created: torch.Size([264, 264])
Train: 2369537 sequences
Val: 251674 sequences
Test: 868533 sequences
Creating adjacency matrix from method3 clustering...
✓ Adjacency matrix: torch.Size([264, 264])

TRAINING SSTZIP-GNN: METHOD3

[Config] Sequence length: 96
[Config] Batch size: 64
[Config] Learning rate: 0.001
[Config] Epochs: 100
[1/7] Getting data loaders for method3...
✓ DataLoaders ready
  - Train batches: 37025
  - Val batches: 3933
  - Test batches: 13571
  - Adjacency matrix: torch.Size([264, 264])

[2/7] Initializing SSTZIP-GNN model...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



❌ Error in experiment method3: float() argument must be a string or a real number, not 'Timestamp'

❌ Error training method3: float() argument must be a string or a real number, not 'Timestamp'

TRAINING SUMMARY

❌ No experiments completed successfully!

✅ PHASE 4 TRAINING COMPLETE



Traceback (most recent call last):
  File "<string>", line 82, in run_experiment
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 741, in __next__
    data = self._next_data()
           ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 801, in _next_data
    data = self._dataset_fetcher.fetch(index)  # may raise StopIteration
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/content/taxi-demand-prediction/src/data/data_loader.py", line 101, in __getitem__
    x = seq_df[self.feature_cols].values.astype(np.float32)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: float() argument must be a string or a real number, not 'Timestamp'
Traceback (most recent call last):
  File "<stri

## 6. Display and Save Results

In [ ]:
import json
from pathlib import Path

print("[Results] Loading training summary...\n")

# Check multiple possible summary locations
summary_paths = [
    Path('logs/experiment_results.json'),
    Path('checkpoints/training_summary.json'),
]

results = None
for path in summary_paths:
    if path.exists():
        with open(path) as f:
            results = json.load(f)
        print(f"[OK] Found results at: {path}\n")
        break

if results:
    print("="*80)
    print("TRAINING RESULTS")
    print("="*80)
    print(json.dumps(results, indent=2))
else:
    print("[INFO] No summary file found yet. Check logs/ directory for individual experiment results.")
    print("\nLooking for experiment results...")
    logs_dir = Path('logs')
    if logs_dir.exists():
        results_files = list(logs_dir.glob('*_results.json'))
        if results_files:
            print(f"\nFound {len(results_files)} result file(s):")
            for f in results_files:
                print(f"  - {f.name}")

In [ ]:
# Display results as table if available
if results:
    print("\n" + "-"*80)
    print("METRICS SUMMARY")
    print("-"*80)
    
    if isinstance(results, dict) and 'results' in results:
        # Summary format from training_summary.json
        print(f"{'Method':<15} {'MAE':<12} {'RMSE':<12} {'MAPE':<12} {'Time (s)':<12}")
        print("-"*63)
        for method, metrics in results['results'].items():
            mae = metrics.get('MAE', 0)
            rmse = metrics.get('RMSE', 0)
            mape = metrics.get('MAPE', 0)
            time_s = metrics.get('Training_Time_Sec', 0)
            print(f"{method:<15} {mae:<12.4f} {rmse:<12.4f} {mape:<12.4f} {time_s:<12.1f}")
    else:
        # Other format
        print(json.dumps(results, indent=2))

## 7. Save Results to Google Drive (if on Colab)

In [ ]:
if IN_COLAB:
    import shutil
    from pathlib import Path
    
    print("[Sync] Copying results to Google Drive...")
    
    # Create checkpoints folder in Google Drive if it doesn't exist
    drive_checkpoints = Path('/content/drive/MyDrive/checkpoints')
    drive_checkpoints.mkdir(parents=True, exist_ok=True)
    
    # Copy local checkpoints
    local_checkpoints = Path('checkpoints')
    if local_checkpoints.exists():
        for method_dir in local_checkpoints.glob('method*'):
            dst = drive_checkpoints / method_dir.name
            if dst.exists():
                shutil.rmtree(dst)
            try:
                shutil.copytree(method_dir, dst)
                print(f"  [OK] {method_dir.name} → Google Drive")
            except Exception as e:
                print(f"  [ERROR] {method_dir.name}: {e}")
    
    # Copy logs
    drive_logs = Path('/content/drive/MyDrive/logs')
    drive_logs.mkdir(parents=True, exist_ok=True)
    
    local_logs = Path('logs')
    if local_logs.exists():
        for log_file in local_logs.glob('*.json'):
            try:
                shutil.copy(log_file, drive_logs / log_file.name)
            except:
                pass
    
    print("\n[OK] Results synced to Google Drive")
else:
    print("[Info] Running locally - results saved to:")
    print("  - checkpoints/  (model checkpoints)")
    print("  - logs/         (training logs & results)")

## Summary

In [ ]:
print("\n" + "="*80)
print("TRAINING WORKFLOW COMPLETE")
print("="*80)

print("\n[Results] Output files:")
print("  - Checkpoints: checkpoints/<method>/<epoch-val_loss>.pt")
print("  - Metrics:     logs/*.json")

print("\n[Next Steps]")
print("  1. Review training results above")
print("  2. Run Phase 5 evaluation script: scripts/09_evaluation.py")
print("  3. Run Phase 6 MongoDB setup: scripts/test_mongodb_connection.py")

if IN_COLAB:
    print("\n[Files] Access results in Google Drive:")
    print("  - Checkpoints: checkpoints/ folder")
    print("  - Logs:        logs/ folder")
else:
    print("\n[Files] All results are in the project directory")

print("\n✅ Training complete!")